# NumericVerifier ML Decision Model (v2, P&L-only)

This notebook trains the supervised ML model to predict **ACCEPT / REPAIR / FLAG** from verifier signals.

- **Dataset:** `runs/signals_v2.csv` (schema v2; each row from a real LLM + verifier run).
- **Features:** All 14 signal columns (including P&L fields); no raw text, evidence, or answers.
- **Exports:** Artifacts are written to `runs/` for the backend (`decision_model_v2.joblib`, `feature_schema_v2.json`, `label_mapping_v2.json`, `ml_metrics_v2.json`).

**Model updates:** Re-run this notebook from top to bottom to retrain and refresh the artifacts the backend uses when `USE_ML_DECIDER=true`.


## 1. Setup and Imports


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Imports successful")


## 2. Load Dataset


In [ ]:
# Upload signals.csv to Colab or mount Google Drive
# Option 1: Upload file directly
# from google.colab import files
# uploaded = files.upload()

# Option 2: Mount Google Drive (if file is in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# data_path = '/content/drive/MyDrive/signals.csv'

# Option 3: Use local path (for testing outside Colab)
# data_path = '../runs/signals.csv'

# Path: runs/signals_v2.csv (v2 schema). For Colab, upload and set data_path.
from pathlib import Path
_root = Path.cwd() if (Path.cwd() / "runs" / "signals_v2.csv").exists() else Path.cwd().parent
data_path = _root / "runs" / "signals_v2.csv"
if not Path(data_path).exists():
    data_path = 'signals_v2.csv'
df = pd.read_csv(data_path)

FEATURE_COLS_V2 = [
    'unsupported_claims_count', 'coverage_ratio', 'recomputation_fail_count',
    'max_relative_error', 'mean_relative_error', 'scale_mismatch_count',
    'period_mismatch_count', 'ambiguity_count', 'schema_version',
    'pnl_table_detected', 'pnl_identity_fail_count', 'pnl_margin_fail_count',
    'pnl_missing_baseline_count', 'pnl_period_strict_mismatch_count'
]
assert set(FEATURE_COLS_V2) <= set(df.columns) and 'decision' in df.columns
print(f"Dataset shape: {df.shape}"); print(df.head())


## 3. Data Exploration


In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Class distribution
print("\nClass distribution:")
class_counts = df['decision'].value_counts()
print(class_counts)
print(f"\nClass proportions:")
print(class_counts / len(df))

# Visualize class distribution
plt.figure(figsize=(8, 5))
class_counts.plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Class Distribution')
plt.xlabel('Decision')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Feature statistics (v2: all 14 signal columns)
print("\nFeature statistics:"); print(df[FEATURE_COLS_V2].describe())


## 4. Preprocessing


In [ ]:
# Features = v2 signal columns (order must match backend feature_schema_v2.json)
X = df[FEATURE_COLS_V2].copy()
y = df['decision'].copy()

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

# Encode labels: ACCEPT → 0, REPAIR → 1, FLAG → 2
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} → {i}")

print(f"\nEncoded label distribution:")
unique, counts = np.unique(y_encoded, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label} ({label_encoder.inverse_transform([label])[0]}): {count}")


In [ ]:
# Stratified train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"\nFeatures scaled (mean=0, std=1)")
print(f"Training set scaled shape: {X_train_scaled.shape}")


## 5. Train Models


In [ ]:
# Model 1: Logistic Regression (multinomial, interpretable baseline)
print("Training Logistic Regression...")

lr_model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

print("✓ Logistic Regression trained")


In [ ]:
# Model 2: Random Forest (non-linear decision boundaries)
print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

print("✓ Random Forest trained")


## 6. Evaluate Models


In [ ]:
def evaluate_model(model, X_val, y_val, model_name, label_encoder):
    """Evaluate model and return metrics."""
    # Predictions
    y_pred = model.predict(X_val)
    
    # Metrics
    accuracy = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average='macro')
    
    # Per-class metrics
    class_report = classification_report(
        y_val, y_pred,
        target_names=label_encoder.classes_,
        output_dict=True
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_val, y_pred)
    
    # Per-class recall (especially FLAG)
    per_class_recall = {}
    for i, label in enumerate(label_encoder.classes_):
        per_class_recall[label] = class_report[label]['recall']
    
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'confusion_matrix': cm,
        'per_class_recall': per_class_recall,
        'classification_report': class_report,
        'predictions': y_pred
    }

# Evaluate both models
print("Evaluating models...")
lr_results = evaluate_model(lr_model, X_val_scaled, y_val, "Logistic Regression", label_encoder)
rf_results = evaluate_model(rf_model, X_val_scaled, y_val, "Random Forest", label_encoder)

print("✓ Evaluation complete")


In [ ]:
# Print metrics comparison
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

print(f"\n{'Metric':<25} {'Logistic Regression':<20} {'Random Forest':<20}")
print("-" * 60)
print(f"{'Accuracy':<25} {lr_results['accuracy']:<20.4f} {rf_results['accuracy']:<20.4f}")
print(f"{'Macro F1-score':<25} {lr_results['macro_f1']:<20.4f} {rf_results['macro_f1']:<20.4f}")

print(f"\n{'Per-Class Recall (Logistic Regression):'}")
for label, recall in lr_results['per_class_recall'].items():
    print(f"  {label}: {recall:.4f}")

print(f"\n{'Per-Class Recall (Random Forest):'}")
for label, recall in rf_results['per_class_recall'].items():
    print(f"  {label}: {recall:.4f}")

# Detailed classification reports
print(f"\n{'=' * 60}")
print("Logistic Regression - Classification Report")
print("=" * 60)
print(classification_report(y_val, lr_results['predictions'], target_names=label_encoder.classes_))

print(f"\n{'=' * 60}")
print("Random Forest - Classification Report")
print("=" * 60)
print(classification_report(y_val, rf_results['predictions'], target_names=label_encoder.classes_))


In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression confusion matrix
disp_lr = ConfusionMatrixDisplay(
    confusion_matrix=lr_results['confusion_matrix'],
    display_labels=label_encoder.classes_
)
disp_lr.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Logistic Regression')

# Random Forest confusion matrix
disp_rf = ConfusionMatrixDisplay(
    confusion_matrix=rf_results['confusion_matrix'],
    display_labels=label_encoder.classes_
)
disp_rf.plot(ax=axes[1], cmap='Blues', values_format='d')
axes[1].set_title('Random Forest')

plt.tight_layout()
plt.show()


## 7. Select Best Model


In [ ]:
# Select best model based on macro F1-score (primary metric)
if rf_results['macro_f1'] > lr_results['macro_f1']:
    best_model = rf_model
    best_model_name = 'Random Forest'
    best_results = rf_results
    print(f"Best model: Random Forest (F1={rf_results['macro_f1']:.4f})")
else:
    best_model = lr_model
    best_model_name = 'Logistic Regression'
    best_results = lr_results
    print(f"Best model: Logistic Regression (F1={lr_results['macro_f1']:.4f})")

print(f"\nBest model accuracy: {best_results['accuracy']:.4f}")
print(f"Best model macro F1: {best_results['macro_f1']:.4f}")


## 8. Export Artifacts


In [ ]:
# Create pipeline (scaler + classifier) and export to runs/ for backend
from sklearn.pipeline import Pipeline

# Export to same dir as data (runs/) when path is under runs/; else cwd
RUNS_DIR = data_path.parent if isinstance(data_path, Path) and 'runs' in str(data_path) else (Path.cwd() / 'runs' if (Path.cwd() / 'runs').exists() else Path.cwd())

decision_pipeline = Pipeline([('scaler', scaler), ('classifier', best_model)])
joblib.dump(decision_pipeline, RUNS_DIR / 'decision_model_v2.joblib')
print("✓ Exported runs/decision_model_v2.joblib")

label_mapping_v2 = {
    'index_to_label': {str(i): str(l) for i, l in enumerate(label_encoder.classes_)},
    'label_to_index': {str(l): i for i, l in enumerate(label_encoder.classes_)},
    'classes': label_encoder.classes_.tolist()
}
with open(RUNS_DIR / 'label_mapping_v2.json', 'w') as f:
    json.dump(label_mapping_v2, f, indent=2)
print("✓ Exported runs/label_mapping_v2.json")


In [ ]:
# Export feature_schema_v2.json (order must match backend)
feature_schema = {
    'feature_names': FEATURE_COLS_V2,
    'feature_order': FEATURE_COLS_V2,
    'n_features': len(FEATURE_COLS_V2),
    'schema_version': 2
}
with open(RUNS_DIR / 'feature_schema_v2.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)
print("✓ Exported runs/feature_schema_v2.json")


In [ ]:
# Export ml_metrics_v2.json to runs/ (for backend / IPD)
ml_metrics_v2 = {
    'model_name': best_model_name,
    'accuracy': float(best_results['accuracy']),
    'macro_f1': float(best_results['macro_f1']),
    'per_class_recall': {k: float(v) for k, v in best_results['per_class_recall'].items()},
    'confusion_matrix': best_results['confusion_matrix'].tolist(),
    'class_labels': label_encoder.classes_.tolist(),
    'n_train': int(len(X_train)), 'n_val': int(len(X_val)),
    'class_distribution': class_counts.to_dict(), 'random_state': 42, 'test_size': 0.2
}
with open(RUNS_DIR / 'ml_metrics_v2.json', 'w') as f:
    json.dump(ml_metrics_v2, f, indent=2)
print("✓ Exported runs/ml_metrics_v2.json")


## 9. Summary

**Exported to `runs/` (v2):**
1. `decision_model_v2.joblib` - Pipeline (scaler + classifier)
2. `label_mapping_v2.json` - Index ↔ ACCEPT/REPAIR/FLAG
3. `feature_schema_v2.json` - Ordered feature list
4. `ml_metrics_v2.json` - Metrics and confusion matrix

**Model updates:** Re-run this notebook from top to bottom to retrain; artifacts in `runs/` are what the backend uses when `USE_ML_DECIDER=true`.
